In [ ]:
# PARAMETERS (for papermill)

# Đường dẫn dữ liệu đã làm sạch từ notebook 01
CLEANED_DATA_PATH = "data/processed/cleaned_data.parquet"

# Thư mục lưu kết quả
OUTPUT_DIR = "data/processed"

# Tên file output
ANOMALY_OUTPUT_FILENAME = "anomalies_detected.csv"
DATA_WITH_ANOMALY_FILENAME = "data_with_anomaly_flags.parquet"

# Phương pháp phát hiện: 'zscore', 'iqr', 'both'
DETECTION_METHOD = "both"

# Ngưỡng cho Z-score (mặc định 3)
ZSCORE_THRESHOLD = 3

# Cột cần phát hiện anomaly
TARGET_COLUMN = "Global_active_power"

# Bật/tắt các biểu đồ
PLOT_ANOMALY_TIMESERIES = True
PLOT_ANOMALY_DISTRIBUTION = True
PLOT_ANOMALY_STATS = True

In [ ]:
# ==================== SETUP ====================
%load_ext autoreload
%autoreload 2

import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Xác định project_root linh hoạt
cwd = os.getcwd()
if os.path.basename(cwd) == "notebooks":
    project_root = os.path.abspath("..")
else:
    project_root = cwd

src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

# Import thư viện tự viết
from energy_forecast_library import DataLoader, AnomalyDetector, Utils

# Tạo thư mục output nếu chưa có
os.makedirs(os.path.join(project_root, OUTPUT_DIR), exist_ok=True)
os.makedirs(os.path.join(project_root, "outputs/figures"), exist_ok=True)

print(f"✅ Project root: {project_root}")
print(f"✅ Output dir: {OUTPUT_DIR}")
print("✅ Đã import thư viện thành công")

In [ ]:
# ==================== LOAD CLEANED DATA ====================
data_path = os.path.join(project_root, CLEANED_DATA_PATH)

if not os.path.exists(data_path):
    print(f"❌ Không tìm thấy file: {data_path}")
    print("Vui lòng chạy notebook 01 trước!")
else:
    df = DataLoader.load_processed_data(data_path)
    
    print(f"✅ Đã tải dữ liệu từ: {CLEANED_DATA_PATH}")
    print(f"📊 Kích thước: {df.shape[0]:,} dòng, {df.shape[1]} cột")
    print(f"📅 Thời gian: {df.index.min()} → {df.index.max()}")
    
    df.head()

In [ ]:
# ==================== PHÁT HIỆN ANOMALY ====================
print(f"🔍 Phát hiện bất thường trên cột: {TARGET_COLUMN}")
print(f"📊 Phương pháp: {DETECTION_METHOD}, Z-score threshold: {ZSCORE_THRESHOLD}\n")

# Phát hiện anomaly
df_with_anomalies = AnomalyDetector.mark_anomalies(
    df, 
    column=TARGET_COLUMN, 
    method=DETECTION_METHOD,
    zscore_threshold=ZSCORE_THRESHOLD
)

# Thống kê
n_anomalies = df_with_anomalies['is_anomaly'].sum()
total_records = len(df_with_anomalies)
anomaly_percent = (n_anomalies / total_records) * 100

print("="*50)
print("📊 KẾT QUẢ PHÁT HIỆN")
print("="*50)
print(f"✅ Tổng số bản ghi: {total_records:,}")
print(f"⚠️ Số điểm bất thường: {n_anomalies:,} ({anomaly_percent:.4f}%)")
print(f"✅ Số điểm bình thường: {total_records - n_anomalies:,} ({100-anomaly_percent:.4f}%)")
print("="*50)

# So sánh giữa các phương pháp nếu dùng 'both'
if DETECTION_METHOD == 'both':
    n_zscore = df_with_anomalies['is_anomaly_zscore'].sum()
    n_iqr = df_with_anomalies['is_anomaly_iqr'].sum()
    n_both = ((df_with_anomalies['is_anomaly_zscore']) & (df_with_anomalies['is_anomaly_iqr'])).sum()
    
    print("\n📊 Chi tiết theo phương pháp:")
    print(f"  - Z-score phát hiện: {n_zscore:,} điểm")
    print(f"  - IQR phát hiện: {n_iqr:,} điểm")
    print(f"  - Cả hai cùng phát hiện: {n_both:,} điểm")

In [ ]:
# ==================== XEM CÁC ĐIỂM ANOMALY ====================
anomalies_df = df_with_anomalies[df_with_anomalies['is_anomaly']].copy()

print(f"📋 Hiển thị 10 điểm bất thường đầu tiên:")
anomalies_df[TARGET_COLUMN].head(10)

In [ ]:
# ==================== THỐNG KÊ ANOMALY ====================
if PLOT_ANOMALY_STATS:
    # So sánh statistics giữa normal và anomaly
    normal_stats = df_with_anomalies[~df_with_anomalies['is_anomaly']][TARGET_COLUMN].describe()
    anomaly_stats = anomalies_df[TARGET_COLUMN].describe()
    
    stats_comparison = pd.DataFrame({
        'Bình thường': normal_stats,
        'Bất thường': anomaly_stats
    })
    
    print("📊 So sánh thống kê giữa normal và anomaly:")
    stats_comparison

In [ ]:
# ==================== BIỂU ĐỒ PHÂN PHỐI ====================
if PLOT_ANOMALY_DISTRIBUTION:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram phân phối
    axes[0].hist(df_with_anomalies[~df_with_anomalies['is_anomaly']][TARGET_COLUMN], 
                 bins=50, alpha=0.7, label='Bình thường', color='blue', edgecolor='black')
    axes[0].hist(anomalies_df[TARGET_COLUMN], 
                 bins=20, alpha=0.7, label='Bất thường', color='red', edgecolor='black')
    axes[0].set_xlabel(TARGET_COLUMN)
    axes[0].set_ylabel('Tần suất')
    axes[0].set_title('Phân phối: Bình thường vs Bất thường')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Boxplot
    data_to_plot = [df_with_anomalies[~df_with_anomalies['is_anomaly']][TARGET_COLUMN].dropna(),
                    anomalies_df[TARGET_COLUMN].dropna()]
    bp = axes[1].boxplot(data_to_plot, labels=['Bình thường', 'Bất thường'], 
                         patch_artist=True)
    bp['boxes'][0].set_facecolor('blue')
    bp['boxes'][1].set_facecolor('red')
    axes[1].set_ylabel(TARGET_COLUMN)
    axes[1].set_title('Boxplot: Bình thường vs Bất thường')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(os.path.join(project_root, 'outputs/figures/anomaly_distribution.png'), 
                dpi=100, bbox_inches='tight')
    plt.show()

In [ ]:
# ==================== BIỂU ĐỒ TIME SERIES VỚI ANOMALY ====================
if PLOT_ANOMALY_TIMESERIES:
    # Lấy mẫu 1 tháng để dễ nhìn (có thể điều chỉnh)
    sample_start = '2007-01-01'
    sample_end = '2007-01-31'
    
    sample_data = df_with_anomalies[sample_start:sample_end]
    sample_anomalies = sample_data[sample_data['is_anomaly']]
    
    fig, axes = plt.subplots(2, 1, figsize=(15, 8))
    
    # Plot 1: Toàn bộ tháng 1
    axes[0].plot(sample_data.index, sample_data[TARGET_COLUMN], 
                 color='blue', linewidth=0.8, label='Dữ liệu')
    axes[0].scatter(sample_anomalies.index, sample_anomalies[TARGET_COLUMN], 
                    color='red', s=30, label='Bất thường', zorder=5)
    axes[0].set_ylabel(TARGET_COLUMN)
    axes[0].set_title(f'Phát hiện bất thường - Tháng 01/2007')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Zoom vào 1 tuần
    week_data = df_with_anomalies['2007-01-08':'2007-01-14']
    week_anomalies = week_data[week_data['is_anomaly']]
    
    axes[1].plot(week_data.index, week_data[TARGET_COLUMN], 
                 color='blue', linewidth=1, label='Dữ liệu')
    axes[1].scatter(week_anomalies.index, week_anomalies[TARGET_COLUMN], 
                    color='red', s=50, label='Bất thường', zorder=5)
    axes[1].set_ylabel(TARGET_COLUMN)
    axes[1].set_title(f'Zoom: Tuần 08-14/01/2007')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig(os.path.join(project_root, 'outputs/figures/anomaly_timeseries.png'), 
                dpi=100, bbox_inches='tight')
    plt.show()

In [ ]:
# ==================== PHÂN TÍCH ANOMALY THEO THỜI GIAN ====================
# Tạo các cột thời gian để phân tích
df_with_anomalies['hour'] = df_with_anomalies.index.hour
df_with_anomalies['dayofweek'] = df_with_anomalies.index.dayofweek
df_with_anomalies['month'] = df_with_anomalies.index.month
df_with_anomalies['year'] = df_with_anomalies.index.year

# Thống kê anomaly theo giờ
anomaly_by_hour = df_with_anomalies.groupby('hour')['is_anomaly'].mean() * 100
anomaly_by_dow = df_with_anomalies.groupby('dayofweek')['is_anomaly'].mean() * 100
anomaly_by_month = df_with_anomalies.groupby('month')['is_anomaly'].mean() * 100

dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Theo giờ
axes[0].bar(anomaly_by_hour.index, anomaly_by_hour.values, color='red', alpha=0.7)
axes[0].set_xlabel('Giờ trong ngày')
axes[0].set_ylabel('Tỷ lệ anomaly (%)')
axes[0].set_title('Tỷ lệ bất thường theo giờ')
axes[0].grid(True, alpha=0.3, axis='y')

# Theo thứ
axes[1].bar(dow_labels, anomaly_by_dow.values, color='red', alpha=0.7)
axes[1].set_xlabel('Thứ')
axes[1].set_ylabel('Tỷ lệ anomaly (%)')
axes[1].set_title('Tỷ lệ bất thường theo thứ')
axes[1].grid(True, alpha=0.3, axis='y')

# Theo tháng
axes[2].bar(month_labels, anomaly_by_month.values, color='red', alpha=0.7)
axes[2].set_xlabel('Tháng')
axes[2].set_ylabel('Tỷ lệ anomaly (%)')
axes[2].set_title('Tỷ lệ bất thường theo tháng')
axes[2].grid(True, alpha=0.3, axis='y')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(project_root, 'outputs/figures/anomaly_by_time.png'), 
            dpi=100, bbox_inches='tight')
plt.show()

print("📈 Phát hiện:")
print(f"- Giờ có tỷ lệ anomaly cao nhất: {anomaly_by_hour.idxmax()}h ({anomaly_by_hour.max():.2f}%)")
print(f"- Giờ có tỷ lệ anomaly thấp nhất: {anomaly_by_hour.idxmin()}h ({anomaly_by_hour.min():.2f}%)")
print(f"- Thứ có tỷ lệ anomaly cao nhất: {dow_labels[anomaly_by_dow.idxmax()]} ({anomaly_by_dow.max():.2f}%)")
print(f"- Tháng có tỷ lệ anomaly cao nhất: {month_labels[anomaly_by_month.idxmax()-1]} ({anomaly_by_month.max():.2f}%)")

In [ ]:
# ==================== PHÂN TÍCH ANOMALY THEO GIÁ TRỊ ====================
# Phân loại anomaly: cao bất thường hay thấp bất thường
mean_normal = df_with_anomalies[~df_with_anomalies['is_anomaly']][TARGET_COLUMN].mean()

anomalies_df = df_with_anomalies[df_with_anomalies['is_anomaly']].copy()
anomalies_df['anomaly_type'] = anomalies_df[TARGET_COLUMN].apply(
    lambda x: 'Cao bất thường' if x > mean_normal else 'Thấp bất thường'
)

anomaly_type_counts = anomalies_df['anomaly_type'].value_counts()

print("📊 Phân loại anomaly:")
for typ, count in anomaly_type_counts.items():
    print(f"  - {typ}: {count} điểm ({count/len(anomalies_df)*100:.2f}%)")

# Vẽ biểu đồ
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
colors = ['orange' if x == 'Cao bất thường' else 'blue' for x in anomalies_df['anomaly_type']]
ax.scatter(range(len(anomalies_df)), anomalies_df[TARGET_COLUMN], 
          c=colors, alpha=0.6, s=20)
ax.axhline(y=mean_normal, color='red', linestyle='--', label=f'Mean normal: {mean_normal:.2f}')
ax.set_xlabel('Index')
ax.set_ylabel(TARGET_COLUMN)
ax.set_title('Phân loại: Cao bất thường vs Thấp bất thường')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(project_root, 'outputs/figures/anomaly_types.png'), 
            dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ==================== QUYẾT ĐỊNH XỬ LÝ ANOMALY ====================
print("="*60)
print("🔍 QUYẾT ĐỊNH XỬ LÝ ANOMALY CHO BÀI TOÁN DỰ BÁO")
print("="*60)

print("""
Có 3 hướng xử lý anomaly:

1. GIỮ NGUYÊN: 
   - Anomaly là một phần của dữ liệu thực tế (ngày lễ, sự kiện đặc biệt)
   - Mô hình cần học được cả những bất thường này
   - Phù hợp nếu anomaly < 1% và không quá cực đoan

2. LOẠI BỎ:
   - Anomaly do lỗi đo đạc, nhiễu
   - Loại bỏ để mô hình học patterns chính xác hơn
   - Có thể thay thế bằng giá trị nội suy

3. GẮN NHÃN VÀ XỬ LÝ RIÊNG:
   - Tạo feature 'is_anomaly' để mô hình học
   - Hoặc xây dựng mô hình riêng cho anomaly
""")

# Thống kê để ra quyết định
n_anomalies = anomalies_df.shape[0]
total_records = df_with_anomalies.shape[0]
anomaly_percent = (n_anomalies / total_records) * 100

print("\n📊 Căn cứ vào dữ liệu:")
print(f"- Tỷ lệ anomaly: {anomaly_percent:.4f}%")
print(f"- Số lượng anomaly: {n_anomalies:,} điểm")

if anomaly_percent < 0.5:
    print("\n✅ GỢI Ý: Tỷ lệ anomaly rất thấp (< 0.5%)")
    print("   Có thể giữ nguyên hoặc loại bỏ (tùy vào phân tích chuyên sâu)")
elif anomaly_percent < 2:
    print("\n⚠️ GỢI Ý: Tỷ lệ anomaly trung bình (0.5% - 2%)")
    print("   Nên phân tích kỹ và cân nhắc gắn nhãn 'is_anomaly'")
else:
    print("\n❌ GỢI Ý: Tỷ lệ anomaly cao (> 2%)")
    print("   Cần điều chỉnh ngưỡng phát hiện hoặc xử lý đặc biệt")

In [ ]:
# ==================== LƯU KẾT QUẢ ====================
# 1. Lưu danh sách các điểm anomaly
anomaly_output_path = os.path.join(project_root, OUTPUT_DIR, ANOMALY_OUTPUT_FILENAME)
anomalies_df.to_csv(anomaly_output_path)

# 2. Lưu dữ liệu đã gắn flag anomaly
data_with_anomaly_path = os.path.join(project_root, OUTPUT_DIR, DATA_WITH_ANOMALY_FILENAME)
Utils.save_dataframe(df_with_anomalies, data_with_anomaly_path)

print("\n✅ Đã lưu kết quả:")
print(f"- Danh sách anomaly: {ANOMALY_OUTPUT_FILENAME} ({len(anomalies_df):,} dòng)")
print(f"- Dữ liệu với flags: {DATA_WITH_ANOMALY_FILENAME}")
print(f"\n✅ Sẵn sàng cho bước tiếp theo: 03_feature_engineering.ipynb")